# FinRisk AI: Phase 1 Data Engineering and EDA

This notebook loads the raw Home Credit datasets, inspects schema quality, and performs business-oriented exploratory analysis for credit risk modeling.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

DATA_DIR = Path.cwd().parent / 'data' / 'raw'
DATA_DIR = DATA_DIR.resolve()
print('Data directory:', DATA_DIR)

files = [
    'application_train.csv',
    'application_test.csv',
    'bureau.csv',
    'bureau_balance.csv',
    'previous_application.csv',
    'installments_payments.csv',
    'POS_CASH_balance.csv',
    'credit_card_balance.csv'
]

for file in files:
    path = DATA_DIR / file
    if path.exists():
        df = pd.read_csv(path)
        print(f'\n=== {file} ===')
        print('Shape:', df.shape)
        print('Columns:', len(df.columns))
        print('Missing values:', int(df.isna().sum().sum()))
        print('Duplicate rows:', int(df.duplicated().sum()))
        print('Sample columns:', list(df.columns[:10]))
    else:
        print(f'\n=== {file} ===')
        print('File not found')

: 

In [ ]:
train_path = DATA_DIR / 'application_train.csv'
if train_path.exists():
    train_df = pd.read_csv(train_path)
    print('Training shape:', train_df.shape)
    print('Target column:', 'TARGET' in train_df.columns)
    print(train_df['TARGET'].value_counts(normalize=True).to_string())
    print('\nMissing values by column:')
    print(train_df.isna().sum().sort_values(ascending=False).head(15).to_string())
else:
    print('application_train.csv not found. Place the Home Credit files in the data/raw directory first.')

In [ ]:
if 'train_df' in locals():
    numeric_cols = train_df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = train_df.select_dtypes(exclude=['number']).columns.tolist()
    print('Numeric columns:', len(numeric_cols))
    print('Categorical columns:', len(categorical_cols))
    print('\nNumeric summary sample:')
    print(train_df[numeric_cols[:8]].describe().T[['mean','std','min','max']].to_string())

In [ ]:
if 'train_df' in locals():
    # Business-focused EDA questions
    business_features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'DAYS_EMPLOYED', 'DAYS_BIRTH', 'CNT_CHILDREN', 'AMT_ANNUITY']
    for col in business_features:
        if col in train_df.columns:
            print(f'\n--- {col} by TARGET ---')
            print(train_df.groupby('TARGET')[col].mean().round(2).to_string())

In [ ]:
if 'train_df' in locals():
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
    except ImportError as exc:
        print('Plotting libraries not installed:', exc)
    else:
        plt.style.use('seaborn-v0_8')
        train_df['loan_to_income_ratio'] = train_df['AMT_CREDIT'] / train_df['AMT_INCOME_TOTAL'].replace(0, np.nan)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.boxplot(data=train_df, x='TARGET', y='loan_to_income_ratio', ax=axes[0])
        axes[0].set_title('Loan-to-Income Ratio by Default Status')
        sns.histplot(data=train_df, x='AMT_INCOME_TOTAL', hue='TARGET', bins=30, kde=True, ax=axes[1], alpha=0.5)
        axes[1].set_title('Income Distribution by Risk Group')
        plt.tight_layout()
        plt.show()